# MaduraApp — Fine-tuning YOLO26n en Google Colab

Notebook diseñado para entornos con GPU gratuita (Colab T4 / Kaggle P100).
Replica el pipeline CRISP-DM de `scripts/train_model.py` en formato celdas.

**Antes de empezar:**
1. `Runtime → Change runtime type → GPU (T4)`
2. Tener tu `ROBOFLOW_API_KEY` a mano (perfil Roboflow → Settings → API Key)

## 1. Instalar dependencias

In [ ]:
!pip install -q ultralytics roboflow pyyaml

In [ ]:
import torch
from ultralytics import YOLO
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Descargar el dataset desde Roboflow

Reemplaza `workspace`, `project` y `version` por los de tu proyecto Roboflow.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

API_KEY = getpass('Roboflow API key: ')
rf = Roboflow(api_key=API_KEY)
project = rf.workspace('maduraapp-duoc').project('maduraapp-ripeness')
dataset = project.version(1).download('yolov8')
print(f'Dataset descargado en: {dataset.location}')

## 3. data.yaml

Roboflow genera un `data.yaml` automático. Verificamos que las 12 clases estén en el orden esperado por el backend.

In [ ]:
import yaml
with open(f'{dataset.location}/data.yaml') as fh:
    data_cfg = yaml.safe_load(fh)
print('Clases del dataset:')
for i, name in enumerate(data_cfg['names']):
    print(f'  {i:>2} → {name}')

## 4. Entrenamiento — fine-tuning YOLO26n

Hiperparámetros idénticos a `scripts/config.yaml`.

In [ ]:
model = YOLO('yolo26n.pt')

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=80,
    batch=16,
    imgsz=640,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    amp=True,
    patience=15,
    hsv_v=0.4,
    degrees=15.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    project='runs',
    name='maduraapp_v1',
    plots=True,
)

## 5. Evaluación — validar KPI mAP@50 ≥ 0.75

In [ ]:
best_pt = 'runs/maduraapp_v1/weights/best.pt'
model = YOLO(best_pt)
metrics = model.val(data=f'{dataset.location}/data.yaml', split='test', plots=True)

print(f'mAP@50    = {metrics.box.map50:.4f}')
print(f'mAP@50-95 = {metrics.box.map:.4f}')
print(f'Precision = {metrics.box.mp:.4f}')
print(f'Recall    = {metrics.box.mr:.4f}')

TARGET = 0.75
passed = metrics.box.map50 >= TARGET
print(f'\nKPI mAP@50 ≥ {TARGET}: {"✅ APROBADO" if passed else "❌ AJUSTAR HIPERPARÁMETROS"}')

## 6. Descargar best.pt al PC

Una vez descargado, mover a `backend/weights/yolo26n_maduraapp.pt` con `scripts/export_model.py`.

In [ ]:
from google.colab import files
files.download(best_pt)

## 7. (Opcional) Visualizar predicciones sobre el set de test

In [ ]:
import glob, random
from IPython.display import Image, display

test_imgs = glob.glob(f'{dataset.location}/test/images/*.jpg')
samples = random.sample(test_imgs, min(5, len(test_imgs)))

for img_path in samples:
    results = model(img_path)
    for r in results:
        out_path = r.save()
        display(Image(out_path))